# Data Wrangling: Fannie Mae Mortgage Delinquency Risk

For this project, I am using Fannie Mae loan performance data to build a loan-month dataset for delinquency prediction. Each row will represent one loan during one reporting month, rather than treating each mortgage as a single static observation.

The goal of this notebook is to take the raw loan performance files, clean the fields, standardize dates and numeric values, check for missing or duplicate records, and create a processed panel that can be used for EDA and modeling.

The main output is:

`data/processed/loan_month_panel.parquet`

I also create a future delinquency target that flags whether a loan becomes 90 or more days delinquent within the next 6 observed reporting months.

Current Dataset source (tentative to add others if data is insufficient):  
https://capitalmarkets.fanniemae.com/credit-risk-transfer/single-family-credit-risk-transfer/fannie-mae-single-family-loan-performance-data

## 1. Setup

This section imports the required libraries and defines project folders.

In [ ]:
import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
PROJECT_ROOT = Path("..")
DATASET_TAG = "2025q1_full"

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / DATASET_TAG

for path in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PANEL_OUTPUT_PARQUET = PROCESSED_DIR / f"loan_month_panel_{DATASET_TAG}.parquet"
PANEL_OUTPUT_CSV = PROCESSED_DIR / f"loan_month_panel_{DATASET_TAG}.csv"
DATA_DICTIONARY_OUTPUT = PROCESSED_DIR / f"data_dictionary_{DATASET_TAG}.csv"
TARGET_SUMMARY_OUTPUT = PROCESSED_DIR / f"target_summary_{DATASET_TAG}.csv"
LOAN_TIMELINE_OUTPUT = PROCESSED_DIR / f"loan_timeline_gap_check_{DATASET_TAG}.csv"

HORIZON_MONTHS = 6
SERIOUS_DQ_THRESHOLD = 3
TARGET_COL = f"future_serious_dq_{HORIZON_MONTHS}m"

ROW_LIMIT = None

MAX_FILES = None

CHUNKSIZE = None


## 2. Fannie Mae Field Layout

The raw Fannie Mae files can be a little awkward to work with because some files may already have headers while others may not. To make the notebook easier to reuse, I set up a list of expected field names and then clean the column names into a consistent format.

The main fields I care about are the loan identifier, reporting period, delinquency status, loan balance, credit score, DTI, LTV, loan purpose, property information, and monthly payment history.

In [ ]:
KNOWN_FANNIE_FIELDS = [
    "reference_pool_id",
    "loan_identifier",
    "monthly_reporting_period",
    "channel",
    "seller_name",
    "servicer_name",
    "master_servicer",
    "original_interest_rate",
    "current_interest_rate",
    "original_upb",
    "upb_at_issuance",
    "current_actual_upb",
    "original_loan_term",
    "origination_date",
    "first_payment_date",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "remaining_months_to_maturity",
    "maturity_date",
    "original_ltv",
    "original_cltv",
    "number_of_borrowers",
    "dti",
    "borrower_credit_score_at_origination",
    "co_borrower_credit_score_at_origination",
    "first_time_home_buyer_indicator",
    "loan_purpose",
    "property_type",
    "number_of_units",
    "occupancy_status",
    "property_state",
    "msa",
    "zip_code_short",
    "mortgage_insurance_percentage",
    "amortization_type",
    "prepayment_penalty_indicator",
    "interest_only_loan_indicator",
    "interest_only_first_principal_and_interest_payment_date",
    "months_to_amortization",
    "current_loan_delinquency_status",
    "loan_payment_history",
    "modification_flag",
    "mortgage_insurance_cancellation_indicator",
    "zero_balance_code",
    "zero_balance_effective_date",
    "upb_at_time_of_removal",
    "repurchase_date",
    "scheduled_principal_current",
    "total_principal_current",
    "unscheduled_principal_current",
    "last_paid_installment_date",
    "foreclosure_date",
    "disposition_date",
    "foreclosure_costs",
    "property_preservation_and_repair_costs",
    "asset_recovery_costs"
]

## 3. Helper Functions

These functions handle raw file detection, column cleaning, type conversion, date parsing, delinquency parsing, and target construction.

In [ ]:
def clean_column_names(columns):
    cleaned = (
        pd.Index(columns)
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return cleaned.tolist()


def detect_delimiter(file_path, sample_size=10_000):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        sample = f.read(sample_size)

    delimiter_counts = {
        "|": sample.count("|"),
        ",": sample.count(","),
        "\t": sample.count("\t")
    }

    return max(delimiter_counts, key=delimiter_counts.get)


def looks_like_header(first_line):
    text = first_line.strip().lower()
    header_keywords = ["loan", "identifier", "reporting", "period", "delinquency", "seller", "servicer"]

    return any(keyword in text for keyword in header_keywords)


def build_column_names(n_cols):
    names = []

    for i in range(n_cols):
        if i < len(KNOWN_FANNIE_FIELDS):
            names.append(KNOWN_FANNIE_FIELDS[i])
        else:
            names.append(f"field_{i+1:03d}")

    return names


def parse_mmyyyy(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    if value.endswith(".0"):
        value = value[:-2]

    value = value.zfill(6)

    try:
        return pd.to_datetime(value, format="%m%Y")
    except ValueError:
        return pd.to_datetime(value, errors="coerce")


def delinquency_to_numeric(value):
    """
    Convert Fannie Mae Current Loan Delinquency Status to numeric.

    00 means current.
    01 means 30 to 59 days delinquent.
    02 means 60 to 89 days delinquent.
    03 means 90 to 119 days delinquent.
    XX means unknown.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip().upper()

    if text in {"", "XX", "X", "NAN", "NONE"}:
        return np.nan

    try:
        return int(text)
    except ValueError:
        return np.nan


def parse_payment_history_codes(value):
    """
    Split a loan payment history string into 2-character monthly codes.
    """
    if pd.isna(value):
        return []

    text = str(value).strip().upper().replace(" ", "")

    if text == "":
        return []

    if len(text) % 2 != 0:
        text = text[1:]

    return [text[i:i+2] for i in range(0, len(text), 2)]


def payment_history_count_dq(value, threshold=1, lookback=12):
    """
    Count how many months in the recent payment history were delinquent.

    threshold=1 means 30+ days delinquent.
    threshold=2 means 60+ days delinquent.
    threshold=3 means 90+ days delinquent.
    """
    codes = parse_payment_history_codes(value)
    recent_codes = codes[-lookback:]

    numeric_codes = [int(code) for code in recent_codes if code.isdigit()]
    return sum(code >= threshold for code in numeric_codes)


def payment_history_max_dq(value, lookback=12):
    """
    Return the maximum delinquency status in the recent payment history.
    """
    codes = parse_payment_history_codes(value)
    recent_codes = codes[-lookback:]

    numeric_codes = [int(code) for code in recent_codes if code.isdigit()]

    if len(numeric_codes) == 0:
        return np.nan

    return max(numeric_codes)


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

In [ ]:
def read_fannie_file(file_path, row_limit=None):
    """
    Read one Fannie Mae raw file.

    The function supports files with or without headers.
    """
    file_path = Path(file_path)
    delimiter = detect_delimiter(file_path)

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline()

    has_header = looks_like_header(first_line)

    if has_header:
        df_raw = pd.read_csv(
            file_path,
            sep=delimiter,
            nrows=row_limit,
            low_memory=False
        )
        df_raw.columns = clean_column_names(df_raw.columns)
    else:
        preview = pd.read_csv(
            file_path,
            sep=delimiter,
            header=None,
            nrows=1,
            low_memory=False
        )
        names = build_column_names(preview.shape[1])

        df_raw = pd.read_csv(
            file_path,
            sep=delimiter,
            header=None,
            names=names,
            nrows=row_limit,
            low_memory=False
        )

    df_raw["source_file"] = file_path.name
    return df_raw

In [ ]:
def standardize_core_fields(df):
    """
    Clean and standardize the fields needed for EDA and modeling.
    """
    df = df.copy()
    df.columns = clean_column_names(df.columns)

    rename_map = {
        "original_loan_to_value_ratio_ltv": "original_ltv",
        "original_combined_loan_to_value_ratio_cltv": "original_cltv",
        "debt_to_income_dti": "dti",
        "zip3": "zip_code_short",
        "current_delinquency_status": "current_loan_delinquency_status",
        "reporting_period": "monthly_reporting_period",
        "loan_id": "loan_identifier"
    }

    available_rename_map = {
        old: new for old, new in rename_map.items()
        if old in df.columns and new not in df.columns
    }

    df = df.rename(columns=available_rename_map)

    return df


def clean_missing_codes(df):
    """
    Replace common missing-value codes with NaN.
    """
    df = df.copy()

    missing_codes = ["", " ", "NA", "N/A", "NULL", "None", "nan", "NaN"]

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].replace(missing_codes, np.nan)

    return df


def convert_types(df):
    """
    Convert important fields to numeric or datetime types.
    """
    df = df.copy()

    date_cols = [
        "monthly_reporting_period",
        "origination_date",
        "first_payment_date",
        "maturity_date",
        "zero_balance_effective_date",
        "repurchase_date",
        "last_paid_installment_date",
        "foreclosure_date",
        "disposition_date"
    ]

    numeric_cols = [
        "original_interest_rate",
        "current_interest_rate",
        "original_upb",
        "upb_at_issuance",
        "current_actual_upb",
        "original_loan_term",
        "loan_age",
        "remaining_months_to_legal_maturity",
        "remaining_months_to_maturity",
        "original_ltv",
        "original_cltv",
        "number_of_borrowers",
        "dti",
        "borrower_credit_score_at_origination",
        "co_borrower_credit_score_at_origination",
        "number_of_units",
        "mortgage_insurance_percentage",
        "months_to_amortization",
        "upb_at_time_of_removal",
        "scheduled_principal_current",
        "total_principal_current",
        "unscheduled_principal_current",
        "foreclosure_costs",
        "property_preservation_and_repair_costs",
        "asset_recovery_costs"
    ]

    for col in date_cols:
        if col in df.columns:
            df[col] = df[col].map(parse_mmyyyy)

    for col in numeric_cols:
        if col in df.columns:
            df[col] = safe_numeric(df[col])

    if "current_loan_delinquency_status" in df.columns:
        df["current_dq_num"] = df["current_loan_delinquency_status"].map(delinquency_to_numeric)

    return df

In [ ]:
def create_behavior_features(df):
    """
    Create monthly behavior features that are known as of the current reporting month.
    """
    df = df.copy()

    if "current_dq_num" in df.columns:
        df["is_current"] = (df["current_dq_num"] == 0).astype("Int64")
        df["is_30plus_dq"] = (df["current_dq_num"] >= 1).astype("Int64")
        df["is_60plus_dq"] = (df["current_dq_num"] >= 2).astype("Int64")
        df["is_90plus_dq"] = (df["current_dq_num"] >= 3).astype("Int64")

    if {"loan_identifier", "monthly_reporting_period", "current_dq_num"}.issubset(df.columns):
        df = df.sort_values(["loan_identifier", "monthly_reporting_period"])
        df["prior_dq_num"] = df.groupby("loan_identifier")["current_dq_num"].shift(1)
        df["dq_status_change"] = df["current_dq_num"] - df["prior_dq_num"]
        df["dq_status_worsened"] = (df["dq_status_change"] > 0).astype("Int64")

    if {"current_actual_upb", "original_upb"}.issubset(df.columns):
        df["upb_ratio"] = df["current_actual_upb"] / df["original_upb"]
        df["paydown_pct"] = 1 - df["upb_ratio"]

    if "loan_payment_history" in df.columns:
        df["count_30plus_dq_past_12m"] = df["loan_payment_history"].map(
            lambda x: payment_history_count_dq(x, threshold=1, lookback=12)
        )
        df["count_60plus_dq_past_12m"] = df["loan_payment_history"].map(
            lambda x: payment_history_count_dq(x, threshold=2, lookback=12)
        )
        df["max_dq_past_12m"] = df["loan_payment_history"].map(
            lambda x: payment_history_max_dq(x, lookback=12)
        )

    return df


def create_future_serious_dq_target(df, horizon_months=6, serious_threshold=3):
    """
    Create the future serious delinquency target.

    A row receives target=1 if the loan reaches serious delinquency in any of
    the next horizon_months reporting periods. Rows with no future observation
    receive missing targets. Rows that are already seriously delinquent are
    excluded from the early-warning label.
    """
    required_cols = {"loan_identifier", "monthly_reporting_period", "current_dq_num"}
    missing_required = required_cols - set(df.columns)

    if missing_required:
        raise KeyError(f"Missing required columns for target creation: {missing_required}")

    df = df.copy()
    df = df.sort_values(["loan_identifier", "monthly_reporting_period"]).reset_index(drop=True)

    lead_cols = []

    for month_ahead in range(1, horizon_months + 1):
        lead_col = f"dq_lead_{month_ahead}m"
        df[lead_col] = df.groupby("loan_identifier")["current_dq_num"].shift(-month_ahead)
        lead_cols.append(lead_col)

    future_max_dq = df[lead_cols].max(axis=1, skipna=True)
    has_future_observation = df[lead_cols].notna().any(axis=1)

    target_col = f"future_serious_dq_{horizon_months}m"

    df[target_col] = np.where(
        has_future_observation,
        (future_max_dq >= serious_threshold).astype(int),
        np.nan
    )

    # This is an early-warning target, so current serious delinquency is not a valid prediction row.
    already_serious = df["current_dq_num"] >= serious_threshold
    df.loc[already_serious, target_col] = np.nan

    df = df.drop(columns=lead_cols)

    return df

## 4. Find the Raw Files

Before cleaning anything, I first need to make sure the notebook can find the raw Fannie Mae files. I am keeping those files in the `data/raw/` folder so the project stays organized and the original data is not mixed in with the cleaned outputs.

I am starting with a smaller file first so I can check that the loading and cleaning steps work correctly. Once the output looks right, I can run the same process on the rest of the files.

In [ ]:
raw_patterns = [
    str(RAW_DIR / "*.csv"),
    str(RAW_DIR / "*.txt"),
    str(RAW_DIR / "*.dat")
]

raw_files = []

for pattern in raw_patterns:
    raw_files.extend(glob.glob(pattern))

raw_files = sorted(raw_files)

if MAX_FILES is not None:
    raw_files = raw_files[:MAX_FILES]

print("Raw files found:", len(raw_files))

for file in raw_files[:10]:
    print(file)

if len(raw_files) == 0:
    raise FileNotFoundError(
        f"No raw files found in {RAW_DIR}. "
        "Download Fannie Mae files and place extracted CSV/TXT/DAT files there."
    )

## 5. Load Raw Files

This section reads the raw files and combines them into one dataframe.

In [ ]:
raw_frames = []

for file_path in raw_files:
    print(f"Reading: {file_path}")
    temp = read_fannie_file(file_path, row_limit=ROW_LIMIT)
    print("Shape:", temp.shape)
    raw_frames.append(temp)

df_raw = pd.concat(raw_frames, ignore_index=True)

print("Combined raw shape:", df_raw.shape)
display(df_raw.head())

In [ ]:
raw_profile = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": [df_raw[col].dtype for col in df_raw.columns],
    "missing_pct": [df_raw[col].isna().mean() * 100 for col in df_raw.columns],
    "n_unique": [df_raw[col].nunique(dropna=True) for col in df_raw.columns]
})

display(raw_profile.head(80))

## 6. Standardize and Clean

This section standardizes names, converts types, handles missing-value codes, and creates core monthly behavior fields.

In [ ]:
df = standardize_core_fields(df_raw)
df = clean_missing_codes(df)
df = convert_types(df)

print("Cleaned shape:", df.shape)
display(df.head())
display(df.dtypes.head(80))

In [ ]:
required_columns = [
    "loan_identifier",
    "monthly_reporting_period",
    "current_loan_delinquency_status"
]

missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise KeyError(
        f"Missing required columns after standardization: {missing_required}. "
        "Check whether the raw file has headers or whether the field layout needs updating."
    )

print("Required columns are present.")

In [ ]:
before_rows = len(df)
df = df.drop_duplicates()
after_rows = len(df)

print("Rows before exact duplicate removal:", f"{before_rows:,}")
print("Rows after exact duplicate removal:", f"{after_rows:,}")
print("Exact duplicates removed:", f"{before_rows - after_rows:,}")

In [ ]:
duplicate_mask = df.duplicated(subset=["loan_identifier", "monthly_reporting_period"], keep=False)
duplicate_count = duplicate_mask.sum()

print("Duplicate loan-month rows:", f"{duplicate_count:,}")

if duplicate_count > 0:
    display(
        df.loc[duplicate_mask, ["loan_identifier", "monthly_reporting_period", "source_file"]]
        .sort_values(["loan_identifier", "monthly_reporting_period"])
        .head(25)
    )

    df = (
        df.sort_values(["loan_identifier", "monthly_reporting_period", "source_file"])
        .drop_duplicates(subset=["loan_identifier", "monthly_reporting_period"], keep="last")
        .reset_index(drop=True)
    )

    print("Shape after duplicate loan-month cleanup:", df.shape)

In [ ]:
df = df.sort_values(["loan_identifier", "monthly_reporting_period"]).reset_index(drop=True)

df["reporting_year"] = df["monthly_reporting_period"].dt.year
df["reporting_month"] = df["monthly_reporting_period"].dt.month
df["reporting_quarter"] = df["monthly_reporting_period"].dt.to_period("Q").astype(str)

if "origination_date" in df.columns:
    df["origination_year"] = df["origination_date"].dt.year
    df["origination_quarter"] = df["origination_date"].dt.to_period("Q").astype(str)

df["loan_month_key"] = (
    df["loan_identifier"].astype(str)
    + "_"
    + df["monthly_reporting_period"].dt.strftime("%Y_%m")
)

display(df[["loan_identifier", "monthly_reporting_period", "loan_month_key"]].head())

In [ ]:
string_id_cols = [
    "loan_identifier",
    "zip_code_short",
    "msa",
    "property_state"
]

for col in string_id_cols:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

# Keep ZIP3 style values consistently padded if needed.
if "zip_code_short" in df.columns:
    df["zip_code_short"] = df["zip_code_short"].str.replace(r"\.0$", "", regex=True)
    df["zip_code_short"] = df["zip_code_short"].str.zfill(3)

Since this project treats each mortgage as a monthly timeline, I want to check whether loans have gaps between their first and last observed reporting month. This matters because the target depends on looking ahead across future monthly records.

A few gaps may be expected depending on how the raw files are structured, but large gaps could affect the target definition or later time-based features.

In [ ]:
# Check whether each loan has missing months inside its observed timeline.
loan_timeline = (
    df.groupby("loan_identifier")
    .agg(
        first_month=("monthly_reporting_period", "min"),
        last_month=("monthly_reporting_period", "max"),
        observed_months=("monthly_reporting_period", "nunique")
    )
    .reset_index()
)

loan_timeline["first_month_index"] = (
    loan_timeline["first_month"].dt.year * 12
    + loan_timeline["first_month"].dt.month
)

loan_timeline["last_month_index"] = (
    loan_timeline["last_month"].dt.year * 12
    + loan_timeline["last_month"].dt.month
)

loan_timeline["expected_months"] = (
    loan_timeline["last_month_index"]
    - loan_timeline["first_month_index"]
    + 1
)

loan_timeline["missing_months_inside_timeline"] = (
    loan_timeline["expected_months"]
    - loan_timeline["observed_months"]
)

display(loan_timeline["missing_months_inside_timeline"].describe())

display(
    loan_timeline
    .sort_values("missing_months_inside_timeline", ascending=False)
    .head(20)
)

## 7. Missingness and Basic Quality Checks

Before moving into EDA, I want to make sure the cleaned panel is usable. This section checks missing values, duplicate loan-month records, date coverage, and the distribution of delinquency status.

These checks are important because missing values and duplicate monthly records could affect both the target definition and the later model results.

In [ ]:
missing_summary = (
    df.isna()
    .mean()
    .mul(100)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
    .sort_values("missing_pct", ascending=False)
)

display(missing_summary.head(50))

In [ ]:
coverage_summary = pd.DataFrame({
    "metric": [
        "rows",
        "unique_loans",
        "unique_reporting_periods",
        "min_reporting_period",
        "max_reporting_period",
        "duplicate_loan_months_after_cleaning"
    ],
    "value": [
        len(df),
        df["loan_identifier"].nunique(),
        df["monthly_reporting_period"].nunique(),
        df["monthly_reporting_period"].min(),
        df["monthly_reporting_period"].max(),
        df.duplicated(subset=["loan_identifier", "monthly_reporting_period"]).sum()
    ]
})

display(coverage_summary)

In [ ]:
dq_status_summary = (
    df["current_dq_num"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("current_dq_num")
    .reset_index(name="count")
)

dq_status_summary["pct"] = dq_status_summary["count"] / len(df) * 100

display(dq_status_summary)

## 8. Create Monthly Behavior Features

These features use only information available as of the current reporting month. They are safe for EDA and can be used later in modeling.

In [ ]:
df = create_behavior_features(df)

behavior_cols = [
    "current_dq_num",
    "prior_dq_num",
    "dq_status_change",
    "dq_status_worsened",
    "upb_ratio",
    "paydown_pct",
    "count_30plus_dq_past_12m",
    "count_60plus_dq_past_12m",
    "max_dq_past_12m"
]

available_behavior_cols = [col for col in behavior_cols if col in df.columns]

display(df[["loan_identifier", "monthly_reporting_period"] + available_behavior_cols].head(20))

## 9. Create Future Delinquency Target

This creates the target for a 6-month prediction window. A loan-month is positive if the loan reaches 90+ day delinquency in any of the next 6 observed reporting periods.

Rows are set to missing if:

1. There is no future performance window available.
2. The loan is already seriously delinquent in the current month.

In [ ]:
df = create_future_serious_dq_target(
    df,
    horizon_months=HORIZON_MONTHS,
    serious_threshold=SERIOUS_DQ_THRESHOLD
)

target_summary = (
    df[TARGET_COL]
    .value_counts(dropna=False)
    .rename_axis(TARGET_COL)
    .reset_index(name="count")
)

target_summary["pct"] = target_summary["count"] / len(df) * 100

display(target_summary)

target_summary.to_csv(TARGET_SUMMARY_OUTPUT, index=False)
print("Saved target summary to:", TARGET_SUMMARY_OUTPUT)

In [ ]:
sample_loans = (
    df.dropna(subset=[TARGET_COL])
    .sample(min(3, df["loan_identifier"].nunique()), random_state=42)["loan_identifier"]
    .unique()
)

timeline_cols = [
    "loan_identifier",
    "monthly_reporting_period",
    "current_loan_delinquency_status",
    "current_dq_num",
    TARGET_COL
]

extra_cols = [
    "loan_age",
    "current_actual_upb",
    "upb_ratio",
    "count_30plus_dq_past_12m",
    "max_dq_past_12m",
    "zero_balance_code"
]

timeline_cols += [col for col in extra_cols if col in df.columns]

display(
    df[df["loan_identifier"].isin(sample_loans)]
    .sort_values(["loan_identifier", "monthly_reporting_period"])
    [timeline_cols]
    .head(100)
)

## 10. Leakage Review

Some fields describe things that happen after a loan has already paid off, been removed, gone through foreclosure, or reached another end state. Those fields are useful for understanding the data, but they should not be used as model inputs.

For example, foreclosure dates, disposition dates, recovery costs, and zero-balance event fields can accidentally tell the model what happens in the future.

In [ ]:
# These are fields that were not mapped to a known Fannie Mae field name.
unmapped_fields = [col for col in df.columns if re.match(r"field_\d+", col)]

unmapped_review = pd.DataFrame({
    "feature": unmapped_fields,
    "missing_pct": [df[col].isna().mean() * 100 for col in unmapped_fields],
    "n_unique": [df[col].nunique(dropna=True) for col in unmapped_fields],
    "recommendation": "exclude_until_mapped"
})

display(unmapped_review)

leakage_candidates = [
    "zero_balance_code",
    "zero_balance_effective_date",
    "upb_at_time_of_removal",
    "repurchase_date",
    "foreclosure_date",
    "disposition_date",
    "foreclosure_costs",
    "property_preservation_and_repair_costs",
    "asset_recovery_costs"
]

leakage_fields_present = [col for col in leakage_candidates if col in df.columns]

leakage_review = pd.DataFrame({
    "feature": leakage_fields_present,
    "recommended_modeling_use": "exclude_from_predictors",
    "reason": "Event, removal, foreclosure, disposition, or recovery information can leak future outcome information."
})

display(leakage_review)

## 11. Save Cleaned Outputs

This section saves the processed loan-month panel. Parquet is preferred because Fannie Mae files are large and parquet preserves types better than CSV.

If parquet support is unavailable, the notebook saves a CSV fallback.

In [ ]:
data_dictionary = pd.DataFrame({
    "feature": df.columns,
    "dtype": [str(df[col].dtype) for col in df.columns],
    "missing_pct": [df[col].isna().mean() * 100 for col in df.columns],
    "n_unique": [df[col].nunique(dropna=True) for col in df.columns]
}).sort_values("feature")

data_dictionary.to_csv(DATA_DICTIONARY_OUTPUT, index=False)
print("Saved data dictionary to:", DATA_DICTIONARY_OUTPUT)

display(data_dictionary.head(50))

In [ ]:
try:
    df.to_parquet(PANEL_OUTPUT_PARQUET, index=False)
    print("Saved processed panel to:", PANEL_OUTPUT_PARQUET)
except Exception as e:
    print("Could not save parquet. Saving CSV instead.")
    print("Parquet error:", e)
    df.to_csv(PANEL_OUTPUT_CSV, index=False)
    print("Saved processed panel to:", PANEL_OUTPUT_CSV)

In [ ]:
loan_timeline.to_csv(LOAN_TIMELINE_OUTPUT, index=False)

print("Saved loan timeline gap check to:")
print(LOAN_TIMELINE_OUTPUT)


In [ ]:
final_summary = pd.DataFrame({
    "metric": [
        "processed_rows",
        "processed_columns",
        "unique_loans",
        "unique_reporting_periods",
        "target_positive_rate",
        "target_valid_rows",
        "output_parquet_exists",
        "output_csv_exists"
    ],
    "value": [
        len(df),
        df.shape[1],
        df["loan_identifier"].nunique(),
        df["monthly_reporting_period"].nunique(),
        df[TARGET_COL].mean(skipna=True),
        df[TARGET_COL].notna().sum(),
        PANEL_OUTPUT_PARQUET.exists(),
        PANEL_OUTPUT_CSV.exists()
    ]
})

display(final_summary)

## 12. Data Wrangling Summary

In this notebook, I loaded one Fannie Mae loan performance file, `2020Q1.csv`, and used a 1,000,000-row sample for the first full cleaning pass. The sample contains 26,503 unique loans across 72 reporting periods, from January 2020 through December 2025.

The notebook standardized the column names, converted date fields into datetime format, converted important balance, credit, and delinquency fields into numeric format, and created a unique loan-month key. I also checked for duplicate records using the loan identifier and monthly reporting period. In this run, there were no duplicate loan-month records after cleaning.

Since the project treats each loan as a monthly timeline, I also checked for missing months inside each loan’s observed history. In this run, the timeline gap check did not show missing months between each loan’s first and last observed month.

I created monthly behavior features such as prior delinquency status, whether delinquency worsened from the previous month, UPB ratio, paydown percentage, and recent delinquency counts from the payment history field. I also created the main target variable, `future_serious_dq_6m`, which flags whether a loan becomes 90 or more days delinquent within the next 6 observed reporting months.

Finally, I separated post-event and leakage-prone fields from normal model candidates. Fields related to zero balance events, foreclosure, disposition, removal, and recovery costs should not be used as standard predictors because they can reveal information that would not be available at the time of prediction.